# 📓 Notebook 1: Thu Thập, Khảo Sát & Chuẩn Bị Tập Dữ Liệu (TCGA-LGG)
**Căn cứ tài liệu:** `docs/KHUNG_NOI_DUNG_TONG_QUAT_DO_AN_XLAS.md`  
* **Quy trình chuẩn Bộ môn:** **Bước 1 — Thu thập & Chuẩn bị tập dữ liệu (Data Preparation)**  
* **Ánh xạ Báo cáo đồ án:** **Mục 3 — Mô tả tập dữ liệu thực nghiệm**  
* **Học phần:** Xử lý ảnh số (2026–2027) — Khoa CNTT — Trường ĐH Công Thương TP.HCM (HUIT)  
* **Nhóm thực hiện:** Hà Mạnh Trường (2045240326) & Huỳnh Tâm Trí (2045240315) — *Trí phụ trách chính*

---

## 🎯 1. Mục tiêu kỹ thuật (Theo Khung Tổng Quát)
1. **Nguồn gốc dữ liệu:** Bộ dữ liệu cộng hưởng từ sọ não The Cancer Genome Atlas Low-Grade Glioma (**TCGA-LGG**), công bố bởi Buda et al. (2019) trên tạp chí *Computers in Biology and Medicine*.
2. **Quy cách dữ liệu:** Ảnh cắt lát MRI chuỗi xung FLAIR/T1-CE, kích thước chuẩn hóa $256 	imes 256$, thang độ xám Grayscale `uint8` $[0, 255]$.
3. **Mặt nạ Ground Truth:** Mặt nạ nhị phân $\{0, 255\}$ được bác sĩ chuyên khoa thần kinh khoanh vùng ổ u não (Core Tumor).
4. **Phân nhóm độ khó:** Khảo sát diện tích u và phân loại thành 3 nhóm (U nhỏ $< 1.5\%$, U trung bình $1.5\% - 5\%$, U lớn $> 5\%$) để kiểm thử tính bền vững của thuật toán xử lý ảnh số.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Thêm đường dẫn gốc dự án để nạp module src
sys.path.append(os.path.abspath(".."))

plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Môi trường sẵn sàng. Đang nạp dữ liệu...")


## 📊 2. Khảo sát Metadata Tập Dữ Liệu
Đọc bảng metadata (`data/full/metadata.csv` hoặc fallback `data/sample/metadata.csv`) chứa thông tin chi tiết về diện tích và tỷ lệ u của các ca bệnh:


In [ ]:
base_dir = "../data/full" if os.path.exists("../data/full/metadata.csv") else "../data/sample"
meta_path = os.path.join(base_dir, "metadata.csv")
df = pd.read_csv(meta_path)
print(f"Nguồn dữ liệu đang nạp: {base_dir}")
print(f"Tổng số lát cắt MRI: {len(df)}")
df[['sample_id', 'patient_id', 'width', 'height', 'tumor_area_pixels', 'tumor_ratio_percent']].head(10)


## 📈 3. Phân Tích Thống Kê Phân Bố Khối U
Đánh giá độ phân tán của diện tích u và tỷ lệ % u trên thiết diện não:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Histogram diện tích pixel
axes[0].hist(df['tumor_area_pixels'], bins=8, color='#3B82F6', edgecolor='black', alpha=0.8)
axes[0].set_title("Phân Bố Diện Tích Khối U (Pixels)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Diện tích (pixels)")
axes[0].set_ylabel("Số lượng ca bệnh")

# 2. Histogram tỷ lệ phần trăm
axes[1].hist(df['tumor_ratio_percent'], bins=8, color='#10B981', edgecolor='black', alpha=0.8)
axes[1].set_title("Phân Bố Tỷ Lệ % U / Thiết Diện Ảnh", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Tỷ lệ diện tích (%)")
axes[1].set_ylabel("Số lượng ca bệnh")

plt.tight_layout()
plt.show()

print("Thống kê mô tả chi tiết:")
print(df[['tumor_area_pixels', 'tumor_ratio_percent']].describe().round(2))


## 🔬 4. Phân Loại 3 Nhóm Ca Bệnh (U Nhỏ, U Vừa, U Lớn)
Căn cứ theo độ khó thuật toán để chuẩn bị kiểm thử ở các bước tiếp theo:


In [ ]:
small_tumors = df[df['tumor_ratio_percent'] < 1.5]
medium_tumors = df[(df['tumor_ratio_percent'] >= 1.5) & (df['tumor_ratio_percent'] <= 5.0)]
large_tumors = df[df['tumor_ratio_percent'] > 5.0]

print(f"- Nhóm U Nhỏ (< 1.5% - Thách thức phát hiện biên): {len(small_tumors)} ca ({list(small_tumors['sample_id'])})")
print(f"- Nhóm U Vừa (1.5% - 5.0% - Độ phân đoạn chuẩn): {len(medium_tumors)} ca ({list(medium_tumors['sample_id'])})")
print(f"- Nhóm U Lớn (> 5.0% - Hiệu ứng chèn ép não thất): {len(large_tumors)} ca ({list(large_tumors['sample_id'])})")


## 🖼️ 5. Trực Quan Hóa Lưới Ảnh MRI & Ground Truth Mask
Hiển thị 6 ca đại diện cho 3 nhóm kích thước u để làm minh chứng cho Báo cáo (Mục 3):


In [ ]:
sample_indices = [0, 4, 6, 8, 10, min(19, len(df)-1)]

fig, axes = plt.subplots(2, len(sample_indices), figsize=(18, 6))

for col_idx, idx in enumerate(sample_indices):
    row = df.iloc[idx]
    img_p = os.path.join(base_dir, "images", row['image_file'])
    mask_p = os.path.join(base_dir, "masks", row['mask_file'])
    
    img = cv2.imread(img_p, cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
    
    # Hàng 1: Ảnh gốc
    axes[0, col_idx].imshow(img, cmap='gray')
    axes[0, col_idx].set_title(f"{row['sample_id']}\n({row['tumor_ratio_percent']}%)", fontsize=10)
    axes[0, col_idx].axis('off')
    
    # Hàng 2: Mặt nạ Ground Truth
    axes[1, col_idx].imshow(mask, cmap='hot')
    axes[1, col_idx].set_title(f"Area: {row['tumor_area_pixels']}px", fontsize=10)
    axes[1, col_idx].axis('off')

plt.suptitle("Một Số Mẫu Ảnh MRI Não & Mặt Nạ Ground Truth Chuyên Gia (TCGA-LGG Benchmark)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
